# QubitX-MaxCut: QAOA for Weighted Max-Cut & SWAP Bottleneck Analysis
**Qiskit Fall Fest 2026 — Phase 1 Hackathon**  
**Team:** QubitX | **Track:** Quantum Optimization | **Problem:** PS-O1 — Max-Cut on Weighted Graphs

---

## 📌 Walkthrough Overview
This notebook presents an end-to-end, fully reproducible demonstration of:
1. **Exact Ground-Truth Max-Cut Baseline** using an exhaustive classical brute-force solver across 4-node and 6-node instances.
2. **Parameterized QAOA Circuit Construction & Optimization** using SciPy COBYLA on `AerSimulator`.
3. **Processor A & Processor B Transpilation Comparison** on documented challenge topologies (`processor_A.json` 5-qubit tree, `processor_B.json` 9-qubit Heavy-Hex).
4. **Transpilation Layout Diagnostics** explaining why Heavy-Hex cyclic connectivity reduces routing SWAPs and CX gate overhead.
5. **Official PS-O1 6-Node Problem Instance Benchmark** (integer weights 1–5) demonstrating hardware capacity limits and actual CX counts.
6. **Graph-Size Capacity & Routing Sweep** across 5, 6, and 7 logical nodes.
7. **Synthetic Noise-Model Evaluation** using QFF 2026 Master Guide calibration ($p_{1q}=0.001$, $p_{2q}=0.005$, $p_{ro}=0.01$) across supported topologies and capacity limits.
8. **Automated Metrics & Figure Generation** saved under `results/tables/` and `results/figures/`.

> **Documented Topologies & Noise Status:** Topologies reflect authoritative specifications from `Processor_AB_Benchmark_Configuration_FINAL.pdf` (5-qubit tree for Processor A, 9-qubit Heavy-Hex for Processor B). Synthetic noise simulation implements the QFF 2026 Master Guide error parameters alongside ideal noiseless simulation. All results are development simulations on `AerSimulator` and do not represent physical hardware execution or official organizer approval.

In [1]:
# 1. Environment Setup & Modular Imports
import sys
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt

from src.graph_utils import (
    create_sample_graph,
    create_pso1_6node_graph,
    solve_max_cut_brute_force,
    create_synthetic_graph,
    calculate_cut_value
)
from src.qaoa_builder import build_qaoa_circuit, optimize_qaoa, get_most_frequent_partition
from src.processors import load_development_processors, inspect_processor_transpilation_layouts
from src.transpilation import DOCUMENTED_BASIS_GATES
from src.noise import build_synthetic_noise_model
from src.benchmarking import (
    run_qaoa_benchmark,
    run_processor_ab_benchmark,
    run_processor_ab_noisy_comparison,
    run_graph_size_routing_sweep
)
from src.visualization import generate_all_plots

print("Environment setup complete. All QubitX modules imported successfully.")


Environment setup complete. All QubitX modules imported successfully.


## 2. Problem Graph & Classical Ground-Truth Baseline

We construct the 4-node, 5-edge weighted development problem graph and compute the exact ground-truth Max-Cut value using exhaustive brute-force evaluation.

In [2]:
g_dev = create_sample_graph()
exact_val, best_part, all_opts = solve_max_cut_brute_force(g_dev)

print(f"Development Graph: {g_dev.number_of_nodes()} nodes, {g_dev.number_of_edges()} edges")
print(f"Exact Ground-Truth Max-Cut Value: {exact_val}")
print(f"Optimal Partition Assignment: {best_part}")


Development Graph: 4 nodes, 5 edges
Exact Ground-Truth Max-Cut Value: 7.0
Optimal Partition Assignment: {0: 0, 1: 1, 2: 0, 3: 0}


## 3. QAOA Circuit Construction & Parameter Optimization

We construct a parameterized QAOA circuit ($p=1$), optimize angles $\gamma$ and $\beta$ using COBYLA on `AerSimulator`, and evaluate expected cut value $\langle C \rangle$.

In [3]:
qc_logical = build_qaoa_circuit(g_dev, p=1, measure=False)
print("Logical QAOA Circuit Depth:", qc_logical.depth())
print("Logical Gate Counts:", dict(qc_logical.count_ops()))

opt_params, opt_exp, counts = optimize_qaoa(g_dev, p=1, maxiter=30, shots=1000, seed=42)
top_part, top_val, prob = get_most_frequent_partition(counts, g_dev)

print(f"Optimized QAOA Expected Cut <C>: {opt_exp:.4f}")
print(f"Approximation Ratio (<C> / C_max): {opt_exp / exact_val:.4%}")
print(f"Most Frequent Sampled Partition Cut: {top_val} (Probability: {prob:.3f})")


Logical QAOA Circuit Depth: 7
Logical Gate Counts: {'rzz': 5, 'h': 4, 'rx': 4}
Optimized QAOA Expected Cut <C>: 6.0060
Approximation Ratio (<C> / C_max): 85.8000%
Most Frequent Sampled Partition Cut: 7.0 (Probability: 0.121)


## 4. Hardware Processor Comparison (Processor A vs Processor B)

We benchmark the same QAOA circuit across documented hardware topologies:
- **Processor A (`processor_A.json`)**: 5-qubit branching tree topology (`[(0,1),(1,2),(1,3),(3,4)]`).
- **Processor B (`processor_B.json`)**: 9-qubit Heavy-Hex topology (`[(0,1),(1,2),(1,6),(2,3),(2,6),(2,5),(3,4),(4,5),(5,6),(6,7),(6,8),(8,0)]`).

We record both high-level routed 2Q gates (RZZ + SWAP) and decomposed hardware native CX gates (`['cx', 'rz', 'sx', 'x']`).

In [4]:
df_ab = run_processor_ab_benchmark(g_dev, p_list=[1, 2], maxiter=25, shots=1000, seed_simulator=42, seed_transpiler=42)
print("=== Processor A & Processor B Benchmark Comparison (4-Node Problem) ===")
cols = ['processor_id', 'physical_qubits', 'qaoa_depth_p', 'orig_depth', 'transpiled_depth', 'orig_2q_gates', 'transpiled_2q_gates', 'swap_overhead', 'transpiled_cx_count', 'opt_expected_cut', 'approx_ratio_expected']
print(df_ab[cols].to_string())


=== Processor A & Processor B Benchmark Comparison (4-Node Problem) ===
  processor_id  physical_qubits  qaoa_depth_p  orig_depth  transpiled_depth  orig_2q_gates  transpiled_2q_gates  swap_overhead  transpiled_cx_count  opt_expected_cut  approx_ratio_expected
0  processor_A                5             1           7                11              5                    7              2                   16             6.006                 0.8580
1  processor_B                9             1           7                 8              5                    6              1                   10             6.006                 0.8580
2  processor_A                5             2          12                18             10                   13              3                   29             5.389                 0.7699
3  processor_B                9             2          12                12             10                   10              0                   20             5.389       

## 5. Transpilation Layout Diagnostics

We inspect the physical qubit mapping selected by Qiskit's layout pass to analyze how coupling cycles in Processor B reduce routing SWAPs.

In [5]:
diag = inspect_processor_transpilation_layouts(qc_logical, seed_transpiler=42)
for pid, info in diag.items():
    print(f"=== {pid} ({info['processor_name']}) ===")
    print("Physical Edges:", info['coupling_edges'])
    print("Initial Layout:", info['initial_layout'])
    print("Transpiled Depth:", info['transpiled_depth'])
    print("Gate Counts:", info['op_counts'])


=== processor_A (Processor A (Challenge Kit 5-Qubit Tree)) ===
Physical Edges: [[0, 1], [1, 2], [1, 3], [3, 4]]
Initial Layout: {'<Qubit register=(4, "q"), index=0>': 1, '<Qubit register=(4, "q"), index=1>': 0, '<Qubit register=(4, "q"), index=2>': 3, '<Qubit register=(4, "q"), index=3>': 2, '<Qubit register=(1, "ancilla"), index=0>': 4}
Transpiled Depth: 11
Gate Counts: {'rzz': 5, 'h': 4, 'rx': 4, 'swap': 2}
=== processor_B (Processor B (Challenge Kit 9-Qubit Heavy-Hex)) ===
Physical Edges: [[0, 1], [1, 2], [1, 6], [2, 3], [2, 6], [2, 5], [3, 4], [4, 5], [5, 6], [6, 7], [6, 8], [8, 0]]
Initial Layout: {'<Qubit register=(4, "q"), index=0>': 3, '<Qubit register=(4, "q"), index=1>': 2, '<Qubit register=(4, "q"), index=2>': 4, '<Qubit register=(4, "q"), index=3>': 6, '<Qubit register=(5, "ancilla"), index=0>': 0, '<Qubit register=(5, "ancilla"), index=1>': 1, '<Qubit register=(5, "ancilla"), index=2>': 5, '<Qubit register=(5, "ancilla"), index=3>': 7, '<Qubit register=(5, "ancilla"), inde

## 6. Official PS-O1 6-Node Instances Benchmark & Capacity Limits

Problem Statement PS-O1 specifies 6-vertex weighted graphs with integer weights 1–5. We independently verify all three instances by exhaustive enumeration and evaluate them on Processors A and B.

In [6]:
for inst in [1, 2, 3]:
    g_inst = create_pso1_6node_graph(instance_id=inst)
    exact_c, best_p, _ = solve_max_cut_brute_force(g_inst)
    print(f"--- PS-O1 Instance {inst} ({g_inst.number_of_nodes()} nodes, {g_inst.number_of_edges()} edges) ---")
    print(f"Exact Max-Cut Ground Truth: {exact_c}")

g_6node = create_pso1_6node_graph(instance_id=1)
df_pso1 = run_processor_ab_benchmark(g_6node, p_list=[1], maxiter=25, shots=1000, seed_simulator=42, seed_transpiler=42, output_csv=None)
print("\n=== PS-O1 6-Node Instance 1 Evaluation on Processors A and B ===")
cols_6 = ['processor_id', 'physical_qubits', 'is_supported', 'transpiled_depth', 'swap_overhead', 'transpiled_cx_count', 'notes']
print(df_pso1[cols_6].to_string())


--- PS-O1 Instance 1 (6 nodes, 8 edges) ---
Exact Max-Cut Ground Truth: 19.0
--- PS-O1 Instance 2 (6 nodes, 9 edges) ---
Exact Max-Cut Ground Truth: 25.0
--- PS-O1 Instance 3 (6 nodes, 7 edges) ---
Exact Max-Cut Ground Truth: 22.0

=== PS-O1 6-Node Instance 1 Evaluation on Processors A and B ===
  processor_id  physical_qubits  is_supported  transpiled_depth  swap_overhead  transpiled_cx_count                                                         notes
0  processor_A                5         False               NaN            NaN                  NaN  Unsupported: logical qubits (6) exceed physical capacity (5)
1  processor_B                9          True              13.0            2.0                 22.0                           Supported development transpilation


## 7. Graph-Size Capacity & Routing Sweep (5, 6, 7 Nodes)

We run a scaling sweep to evaluate physical qubit capacity limits and routing SWAP / CX gate inflation on expanding graph sizes.

In [7]:
df_sweep = run_graph_size_routing_sweep(node_counts=[5, 6, 7], p=1, seed_transpiler=42)
print("=== Graph-Size Routing Sweep Results ===")
cols_sw = ['logical_nodes', 'logical_edges', 'processor_id', 'physical_qubits', 'is_supported', 'transpiled_depth', 'swap_overhead', 'transpiled_cx_count', 'notes']
print(df_sweep[cols_sw].to_string())


=== Graph-Size Routing Sweep Results ===
   logical_nodes  logical_edges processor_id  physical_qubits  is_supported  transpiled_depth  swap_overhead  transpiled_cx_count                                                         notes
0              5              7  processor_A                5          True              11.0            3.0                 23.0                           Supported development transpilation
1              5              7  processor_B                9          True              10.0            1.0                 17.0                           Supported development transpilation
2              6              9  processor_A                5         False               NaN            NaN                  NaN  Unsupported: logical qubits (6) exceed physical capacity (5)
3              6              9  processor_B                9          True              12.0            2.0                 24.0                           Supported development transpilation

## 8. Synthetic Noise-Model Evaluation (Master Guide Calibration)

We evaluate the impact of synthetic physical noise on QAOA expectation values and cut quality retention following the official **Qiskit Fall Fest 2026 Master Guide** specification:
- **1-Qubit Depolarizing Error ($p_{1q} = 0.001$):** Applied to physical single-qubit microwave gates (`sx`, `x`). Virtual $R_Z$ rotations are software frame shifts and modeled error-free.
- **2-Qubit CX Depolarizing Error ($p_{2q} = 0.005$):** Applied to decomposed native `cx` interactions.
- **Readout Bit-Flip Error ($p_{ro} = 0.01$):** Applied symmetrically to measurement operations.

Pre-optimized ideal QAOA parameters are evaluated on transpiled circuits for each supported processor with **4,096 shots** and `seed=42`. Unsupported capacity-limited cases (Processor A on ≥6 nodes) are explicitly preserved.

In [8]:
# 8. Synthetic Noise Evaluation on 4-Node and 6-Node Problems
print("=== 4-Node Problem Noisy Comparison (4096 Shots) ===")
df_noisy_4 = run_processor_ab_noisy_comparison(
    g_dev,
    p_list=[1, 2],
    shots_ideal=1000,
    shots_noisy=4096,
    seed_simulator=42,
    seed_transpiler=42,
    output_csv="results/tables/processor_ab_noisy_comparison.csv",
)
cols_noisy = [
    "processor_id", "qaoa_depth_p", "transpiled_cx_count", "transpiled_depth",
    "opt_expected_cut", "noisy_expected_cut", "noisy_approx_ratio",
    "expectation_drop", "solution_quality_retention"
]
print(df_noisy_4[cols_noisy].to_string())

print("\n=== PS-O1 6-Node Instance 1 Noisy Comparison ===")
g_6node = create_pso1_6node_graph(instance_id=1)
df_noisy_6 = run_processor_ab_noisy_comparison(
    g_6node,
    p_list=[1],
    shots_ideal=1000,
    shots_noisy=4096,
    seed_simulator=42,
    seed_transpiler=42,
    output_csv=None,
)
cols_noisy_6 = [
    "processor_id", "physical_qubits", "is_supported", "transpiled_cx_count",
    "transpiled_depth", "opt_expected_cut", "noisy_expected_cut", "noisy_approx_ratio", "notes"
]
print(df_noisy_6[cols_noisy_6].to_string())


=== 4-Node Problem Noisy Comparison (4096 Shots) ===
  processor_id  qaoa_depth_p  transpiled_cx_count  transpiled_depth  opt_expected_cut  noisy_expected_cut  noisy_approx_ratio  expectation_drop  solution_quality_retention
0  processor_A             1                   16                40             6.006              5.8074              0.8296            0.1986                      0.9669
1  processor_B             1                   10                24             6.006              5.8660              0.8380            0.1400                      0.9767
2  processor_A             2                   29                63             5.389              5.1453              0.7350            0.2437                      0.9548
3  processor_B             2                   20                41             5.389              5.2415              0.7488            0.1475                      0.9726

=== PS-O1 6-Node Instance 1 Noisy Comparison ===
  processor_id  physical_qubits  is_s

## 9. Generate Figures & Output Verification

We run `generate_all_plots()` to export publication-quality figures under `results/figures/` and `results/plots/`.

In [9]:
fig_files = generate_all_plots()
print("Generated Plot Figures:")
for f in fig_files:
    print(" -", f)


Generated Plot Figures:
 - results/figures\qaoa_performance_by_depth.png
 - results/figures\transpilation_depth.png
 - results/figures\swap_gate_overhead.png
 - results/plots\qaoa_performance_by_depth.png
 - results/plots\transpilation_depth.png
 - results/plots\swap_gate_overhead.png
